# Build an Automated AI Red Team Assessment with Counterfit

This demo configures a computer vision model target, executes multiple adversarial attacks with Microsoft Counterfit, and compares assessment results. Counterfit must be installed in the student environment.

In [ ]:
from pathlib import Path
import json
import os
import sys

CURRENT = Path.cwd().resolve()
ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
sys.path.append(str(ROOT / "src"))

os.environ["MPLCONFIGDIR"] = str(ROOT / ".matplotlib")

import torch

sys.path.append(str(ROOT / "targets"))

from counterfit_demo_utils import evaluate_accuracy, require_counterfit, run_counterfit_attack_plan, write_results_csv
from logistics_cifar10_resnet18 import LogisticsCifar10Resnet18

DATA_DIR = ROOT / "data" / "generated"
DOWNLOAD_DIR = ROOT / "data" / "cifar10"
MODEL_PATH = ROOT / "models" / "cifar10_resnet18_demo.pt"
RESULTS_DIR = ROOT / "results"
CONFIG_DIR = ROOT / "configs"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
counterfit, Counterfit = require_counterfit()

print(f"Device: {DEVICE}")
print(f"Counterfit module: {counterfit.__file__}")

## 1. Review the Counterfit-Style Target and Attack Configuration

In [ ]:
target_config = json.loads((CONFIG_DIR / "counterfit_target_config.json").read_text(encoding="utf-8"))
attack_plan = json.loads((CONFIG_DIR / "counterfit_attack_plan.json").read_text(encoding="utf-8"))

target_config, attack_plan

## 2. Load the Counterfit Target

The target class wraps the CIFAR-10 ResNet-18 model behind Counterfit's `CFTarget` interface. The first run downloads CIFAR-10 and trains a compact checkpoint. Later runs reuse the saved checkpoint.

In [ ]:
target = LogisticsCifar10Resnet18()
target.load()

clean_eval = evaluate_accuracy(target.model, target.X, target.y, device=target.device)

print(f"Target name: {target.target_name}")
print(f"Validation images: {len(target.y)}")
print(f"Clean accuracy: {clean_eval['accuracy']:.3f}")
print(f"Mean confidence: {clean_eval['mean_confidence']:.3f}")

## 3. Counterfit Execution Point

Counterfit builds an attack object for each configured attack and executes it against the target's `predict(...)` interface.

In [ ]:
for attack in attack_plan["attacks"]:
    print(f"Configured attack: {attack['name']} -> {attack['parameters']}")

## 4. Execute Multiple Attacks and Compare Results

In [ ]:
rows = run_counterfit_attack_plan(target, attack_plan, sample_index=0)
write_results_csv(rows, RESULTS_DIR / "counterfit_attack_results.csv")
rows

## 5. Inspect Generated Results

In [ ]:
for row in rows:
    print(row)

## 6. Interpret the Assessment

Automated attacks accelerate testing, but the result table is not the final risk decision. Interpret the findings with operational context: which package label classes matter most, whether camera inputs can be manipulated, and whether downstream automation relies on high-confidence predictions.